# TinyGist artifact figure notebook

This lightweight notebook reads the files produced directly by the simulation, QEMU, and physical-device workflows. It does not consume preprocessed `.npy` or `.npz` files.

1. Put each result set under its `results/<claim>/raw/` directory.
2. Enable only the corresponding Boolean flag in the next cell. For Figure 14, also choose its explicit comparison path.
3. Run the notebook from the top.

All claim flags default to `False`. With the defaults, the notebook does not access raw-data paths or create output directories.

In [ ]:
RUN_FIGURE_10 = False
RUN_FIGURE_12 = False
RUN_TABLE_3 = False
RUN_FIGURE_14 = False
RUN_FIGURE_16 = False
RUN_FIGURE_19 = False
RUN_FIGURE_20 = False
RUN_FIGURE_21 = False
RUN_FIGURE_22 = False

# When True, a claim fails clearly if its expected condition grid is incomplete.
STRICT_INPUT = True
# Save PDF, PNG, and summary CSV files under results/<claim>/output/.
EXPORT_OUTPUTS = True

# Figure 10(n) displays the first 70 uYOLO-COCO Person records, matching the paper.
FIGURE_10_UYOLO_PANEL_ID = 14
FIGURE_10_UYOLO_REPORT_ROUNDS = 70

# Figure 14 paths: emulation_only, emulation_simulation, emulation_simulation_real.
FIGURE_14_PATH = 'emulation_only'
FIGURE_14_REPORT_ROUNDS = 200

# Figure 19 in the paper uses the ten-device MobileNetV1-CIFAR-10 condition.
FIGURE_19_PANEL_ID = 7
# Figure 20 reports TX/RX volume for the same representative device used in the paper.
FIGURE_20_DEVICE_INDEX = 0

In [ ]:
from __future__ import annotations

import json
from io import BytesIO
import math
import re
from pathlib import Path
from typing import Iterable

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image as IPythonImage, display

NOTEBOOK_ROOT = Path.cwd().resolve()
if not (NOTEBOOK_ROOT / 'results').is_dir():
    candidate = NOTEBOOK_ROOT / 'result_folder_all'
    if (candidate / 'results').is_dir():
        NOTEBOOK_ROOT = candidate
RESULTS_ROOT = NOTEBOOK_ROOT / 'results'

plt.rcParams.update({
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'legend.fontsize': 8,
    'figure.dpi': 120,
    'savefig.bbox': 'standard',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

METHOD_ORDER = ['Centralized', 'DFA', 'tinyGist', 'SDFA', 'SP']
COMM_METHOD_ORDER = ['DFA', 'tinyGist', 'SDFA', 'SP']
METHOD_COLORS = {
    'Centralized': '#4d4d4d',
    'DFA': '#2f55b7',
    'tinyGist': '#9a0c6a',
    'SDFA': '#159985',
    'SP': '#f28e1c',
}
REFERENCE_COLORS = {
    'Gradient magnitude': '#f28e1c',
    'Gradient-weight': '#174a7e',
    'Empirical Fisher-diagonal': '#9a0c6a',
    'Empirical Fisher-weight': '#159985',
    'Hutchinson-diagonal': '#398b7d',
    'Hutchinson-weight': '#c73232',
}

print(f'Notebook root: {NOTEBOOK_ROOT}')

## Shared raw-data readers

In [ ]:
def nested_get(mapping, path, default=None):
    current = mapping
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def read_yaml(path: Path) -> dict:
    if not path.is_file():
        return {}
    value = yaml.safe_load(path.read_text(encoding='utf-8'))
    return value if isinstance(value, dict) else {}


def claim_root(name: str, *parts: str) -> Path:
    root = RESULTS_ROOT / name / 'raw'
    for part in parts:
        root /= part
    if not root.is_dir():
        raise FileNotFoundError(f'Missing raw-data directory: {root}')
    return root


def output_root(name: str) -> Path:
    target = RESULTS_ROOT / name / 'output'
    if EXPORT_OUTPUTS:
        target.mkdir(parents=True, exist_ok=True)
    return target


def save_figure(fig, claim: str, stem: str, bbox_inches='tight') -> None:
    if EXPORT_OUTPUTS:
        target = output_root(claim)
        fig.savefig(target / f'{stem}.pdf', bbox_inches=bbox_inches)
        fig.savefig(target / f'{stem}.png', dpi=220, bbox_inches=bbox_inches)


def display_figure(fig) -> None:
    """Render and display a figure inline with Agg, then release it."""
    with BytesIO() as image_buffer:
        fig.savefig(image_buffer, format='png', dpi=fig.dpi, bbox_inches='tight')
        display(IPythonImage(data=image_buffer.getvalue()))
    plt.close(fig)


def normalize_method(value: object) -> str:
    token = str(value or '').strip().lower().replace('-', '').replace('_', '')
    aliases = {
        'centralized': 'Centralized',
        'cfl': 'Centralized',
        'dfa': 'DFA',
        'gist': 'tinyGist',
        'tinygist': 'tinyGist',
        'gistada': 'tinyGist',
        'sdfa': 'SDFA',
        'segmentpulling': 'SP',
        'sp': 'SP',
    }
    return aliases.get(token, str(value or '').strip())


def normalize_importance_metric(value: object, source_stem: str = '') -> str:
    token = f'{value} {source_stem}'.lower().replace('-', '_')
    checks = (
        ('gradient_weight', 'Gradient-weight'),
        ('gradient_magnitude', 'Gradient magnitude'),
        ('empirical_fisher_weight', 'Empirical Fisher-weight'),
        ('fisher_empirical_diagonal_weight', 'Empirical Fisher-weight'),
        ('empirical_fisher_diagonal', 'Empirical Fisher-diagonal'),
        ('fisher_empirical_diagonal', 'Empirical Fisher-diagonal'),
        ('hutchinson_weight', 'Hutchinson-weight'),
        ('hutchinson_diagonal_weight', 'Hutchinson-weight'),
        ('hutchinson_diagonal', 'Hutchinson-diagonal'),
        ('parameter_magnitude', 'Parameter magnitude'),
    )
    for needle, label in checks:
        if needle in token:
            return label
    return str(value or source_stem)


def task_label(model: str, dataset: str) -> str:
    dataset_names = {
        'mnist': 'MNIST', 'fmnist': 'Fashion-MNIST', 'muscle_gesture': 'Gesture',
        'cifar10': 'CIFAR-10', 'cifar100': 'CIFAR-100', 'emnist': 'EMNIST',
        'google_speech': 'Speech Commands', 'google_speech_kws': 'KWS',
        'vww': 'Visual Wake Words', 'svhn': 'SVHN',
        'fomo_vehicle': 'COCO Vehicle', 'fomo_vehicle_binary': 'COCO Vehicle',
        'uyolo_person': 'COCO Person', 'yolo_person': 'COCO Person',
    }
    model_names = {
        'FCN': 'FCN', 'BasicConv': 'CNN', 'LeNet5': 'LeNet5',
        'MobileNetV1Small': 'MobileNetV1', 'MobileNetV1': 'MobileNetV1',
        'FOMOMNv2Baseline': 'FOMO',
        'MobileNetV2': 'MobileNetV2', 'MobileNetV2Baseline': 'MobileNetV2',
        'MobileNetV4': 'MobileNetV4', 'MobileNetV4Small': 'MobileNetV4',
        'uYOLO': 'uYOLO',
    }
    return f"{model_names.get(model, model)}–{dataset_names.get(dataset, dataset)}"


def run_context(run_dir: Path, raw_root: Path) -> dict:
    config = read_yaml(run_dir / 'used_config.yml')
    metadata = read_yaml(run_dir / 'run_metadata.yml')
    source_stem = Path(str(metadata.get('source_config_file', ''))).stem
    if not source_stem:
        source_stem = run_dir.name
    panel_match = re.match(r'^(\d+)_', source_stem)
    panel_id = int(panel_match.group(1)) if panel_match else np.nan
    model = str(nested_get(config, ('model', 'name'), metadata.get('model', '')))
    dataset = str(nested_get(config, ('dataset', 'name'), metadata.get('dataset', '')))
    method = normalize_method(nested_get(config, ('method', 'name'), metadata.get('method', '')))
    clients = nested_get(config, ('federation', 'clients', 'count'), np.nan)
    metric = nested_get(config, ('method', 'segment_importance', 'metric'), '')
    planned = nested_get(metadata, ('differential_privacy', 'planned_privacy'), {}) or {}
    actual = nested_get(metadata, ('differential_privacy', 'actual_privacy'), {}) or {}
    planned_epsilon = planned.get('epsilon', np.nan) if isinstance(planned, dict) else np.nan
    actual_epsilon = actual.get('maximum_epsilon', np.nan) if isinstance(actual, dict) else np.nan
    delta = planned.get('delta', np.nan) if isinstance(planned, dict) else np.nan
    if pd.isna(delta):
        delta = nested_get(config, ('differential_privacy', 'delta'), np.nan)
    target_match = re.search(r'_dp_sgd_(0_5|1|2|4|8|20)_1e-5$', source_stem)
    configured_target_epsilon = np.nan
    if target_match:
        configured_target_epsilon = (
            0.5 if target_match.group(1) == '0_5' else float(target_match.group(1))
        )
    return {
        'run_id': str(run_dir.relative_to(raw_root)),
        'source_stem': source_stem,
        'panel_id': panel_id,
        'model': model,
        'dataset': dataset,
        'task': task_label(model, dataset),
        'method': method,
        'client_count': int(clients) if not pd.isna(clients) else np.nan,
        'seed': metadata.get('model_initialization_seed', nested_get(config, ('experiment', 'seed'), np.nan)),
        'importance_metric': normalize_importance_metric(metric, source_stem),
        'epsilon': (
            float(configured_target_epsilon)
            if not pd.isna(configured_target_epsilon) else np.nan
        ),
        'configured_target_epsilon': (
            float(configured_target_epsilon)
            if not pd.isna(configured_target_epsilon) else np.nan
        ),
        'planned_epsilon': (
            float(planned_epsilon) if not pd.isna(planned_epsilon) else np.nan
        ),
        'actual_epsilon': (
            float(actual_epsilon) if not pd.isna(actual_epsilon) else np.nan
        ),
        'delta': float(delta) if not pd.isna(delta) else np.nan,
    }


def simulation_run_dirs(raw_root: Path, required: str) -> list[Path]:
    return sorted({path.parent for path in raw_root.rglob(required) if path.is_file()})


def read_metric_frame(run_dir: Path) -> pd.DataFrame:
    workbook = run_dir / 'metrics.xlsx'
    sheet_candidates = ('test_accuracy_post_agg', 'test_acc_after')
    if workbook.is_file():
        excel = pd.ExcelFile(workbook)
        for sheet in sheet_candidates:
            if sheet in excel.sheet_names:
                return pd.read_excel(excel, sheet_name=sheet, index_col=0)
    fallback = run_dir / 'metrics'
    for sheet in sheet_candidates:
        csv_path = fallback / f'{sheet}.csv'
        if csv_path.is_file():
            return pd.read_csv(csv_path, index_col=0)
    raise FileNotFoundError(f'No supported test-accuracy table in {run_dir}')


def read_simulation_accuracy(raw_root: Path) -> pd.DataFrame:
    frames = []
    for run_dir in simulation_run_dirs(raw_root, 'used_config.yml'):
        try:
            frame = read_metric_frame(run_dir)
        except FileNotFoundError:
            continue
        frame.index = pd.to_numeric(frame.index, errors='coerce')
        frame = frame.loc[frame.index.notna()].apply(pd.to_numeric, errors='coerce')
        long = frame.rename_axis('round').reset_index().melt(
            id_vars='round', var_name='device', value_name='accuracy'
        ).dropna(subset=['accuracy'])
        if long.empty:
            continue
        if long['accuracy'].abs().max() <= 1.5:
            long['accuracy'] *= 100.0
        context = run_context(run_dir, raw_root)
        for key, value in context.items():
            long[key] = value
        frames.append(long)
    if not frames:
        raise FileNotFoundError(f'No complete simulation metric runs found under {raw_root}')
    return pd.concat(frames, ignore_index=True)


def run_accuracy_curves(data: pd.DataFrame) -> pd.DataFrame:
    keys = ['run_id', 'panel_id', 'task', 'model', 'dataset', 'method',
            'client_count', 'importance_metric', 'epsilon', 'delta', 'round']
    return data.groupby(keys, dropna=False, as_index=False)['accuracy'].mean()


def curve_summary(data: pd.DataFrame, condition_keys: list[str]) -> pd.DataFrame:
    per_run = data.groupby(['run_id', *condition_keys, 'round'], dropna=False, as_index=False)['accuracy'].mean()
    return per_run.groupby([*condition_keys, 'round'], dropna=False)['accuracy'].agg(
        mean='mean', sd='std', runs='count'
    ).reset_index()


def final_per_run(data: pd.DataFrame, condition_keys: list[str]) -> pd.DataFrame:
    curves = data.groupby(['run_id', *condition_keys, 'round'], dropna=False, as_index=False)['accuracy'].mean()
    return curves.sort_values('round').groupby(['run_id', *condition_keys], dropna=False, as_index=False).tail(1)


def require_conditions(actual: Iterable, expected: Iterable, label: str) -> None:
    if not STRICT_INPUT:
        return
    missing = sorted(set(expected) - set(actual), key=str)
    if missing:
        raise ValueError(f'{label} is missing conditions: {missing}')

## Figure 10 — task/method convergence

Panel (n) follows the paper/reference plot by showing the first 70 uYOLO–COCO Person records; the other panels retain their complete round ranges.

In [ ]:
if RUN_FIGURE_10:
    root = claim_root('Figure10')
    accuracy = read_simulation_accuracy(root)
    require_conditions(accuracy['panel_id'].dropna().astype(int), range(1, 16), 'Figure 10 panels')
    require_conditions(accuracy['method'], METHOD_ORDER, 'Figure 10 methods')
    uyolo_mask = (
        accuracy['model'].eq('uYOLO')
        & accuracy['dataset'].isin({'yolo_person', 'uyolo_person'})
    )
    uyolo_panels = set(accuracy.loc[uyolo_mask, 'panel_id'].dropna().astype(int))
    expected_uyolo_panels = {FIGURE_10_UYOLO_PANEL_ID}
    if STRICT_INPUT and uyolo_panels != expected_uyolo_panels:
        raise ValueError(
            f'Figure 10 uYOLO panel mismatch: expected {expected_uyolo_panels}, found {uyolo_panels}'
        )
    accuracy_for_plot = accuracy.loc[
        ~uyolo_mask | accuracy['round'].lt(FIGURE_10_UYOLO_REPORT_ROUNDS)
    ].copy()
    summary = curve_summary(accuracy_for_plot, ['panel_id', 'task', 'method'])
    panels = sorted(summary['panel_id'].dropna().astype(int).unique())
    fig, axes = plt.subplots(3, 5, figsize=(13.8, 7.3), sharex=False)
    for axis, panel in zip(axes.flat, panels):
        panel_data = summary[summary['panel_id'] == panel]
        for method in METHOD_ORDER:
            line = panel_data[panel_data['method'] == method].sort_values('round')
            if line.empty:
                continue
            axis.plot(line['round'], line['mean'], color=METHOD_COLORS[method], label=method, lw=1.5)
            if line['sd'].notna().any():
                sd = line['sd'].fillna(0)
                axis.fill_between(line['round'], line['mean'] - sd, line['mean'] + sd,
                                  color=METHOD_COLORS[method], alpha=0.12)
        if panel in uyolo_panels:
            axis.set_xlim(0, FIGURE_10_UYOLO_REPORT_ROUNDS - 1)
            axis.set_xticks(range(0, FIGURE_10_UYOLO_REPORT_ROUNDS, 20))
        axis.set_title(f"({chr(96 + panel)}) {panel_data['task'].iloc[0]}")
        axis.grid(alpha=0.25)
    for axis in axes[-1, :]:
        axis.set_xlabel('Round')
    for axis in axes[:, 0]:
        axis.set_ylabel('Accuracy (%)')
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=5, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_figure(fig, 'Figure10', 'figure10_accuracy')
    display_figure(fig)
else:
    print('Figure 10: disabled')

## Figure 12 — scalability

In [ ]:
if RUN_FIGURE_12:
    root = claim_root('Figure12')
    accuracy = read_simulation_accuracy(root)
    require_conditions(accuracy['client_count'].dropna().astype(int), [5, 10, 30, 70, 100, 200], 'Figure 12 client counts')
    require_conditions(accuracy['method'], METHOD_ORDER, 'Figure 12 methods')
    final = final_per_run(accuracy, ['task', 'method', 'client_count'])
    summary = final.groupby(['task', 'method', 'client_count'])['accuracy'].agg(
        mean='mean', sd='std', runs='count'
    ).reset_index()
    tasks = sorted(summary['task'].unique())
    if STRICT_INPUT and len(tasks) != 2:
        raise ValueError(f'Figure 12 expects two tasks, found {tasks}')
    client_counts = [5, 10, 30, 70, 100, 200]
    x_positions = np.arange(len(client_counts), dtype=float)
    bar_width = 0.16
    fig, axes = plt.subplots(len(tasks), 1, figsize=(5.2, 5.0), squeeze=False, sharex=True)
    for axis, task in zip(axes.flat, tasks):
        task_data = summary[summary['task'] == task]
        for method_index, method in enumerate(METHOD_ORDER):
            bars = task_data[task_data['method'] == method].set_index('client_count').reindex(client_counts)
            if STRICT_INPUT and bars['mean'].isna().any():
                missing = [count for count in client_counts if pd.isna(bars.loc[count, 'mean'])]
                raise ValueError(f'Figure 12 is missing {task}/{method} client counts: {missing}')
            offset = (method_index - (len(METHOD_ORDER) - 1) / 2) * bar_width
            axis.bar(x_positions + offset, bars['mean'], width=bar_width,
                     yerr=bars['sd'].fillna(0), capsize=2,
                     color=METHOD_COLORS[method], label=method)
        axis.set_title(task)
        axis.set_ylabel('Final accuracy (%)')
        axis.set_xticks(x_positions, [str(count) for count in client_counts])
        axis.grid(axis='y', alpha=0.25)
    axes[-1, 0].set_xlabel('Participating devices')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=5, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    save_figure(fig, 'Figure12', 'figure12_scalability')
    display_figure(fig)
else:
    print('Figure 12: disabled')

## Table 3 — importance metrics

This section prints an organized numeric summary. It does not render a publication table.

In [ ]:
def read_score_times(raw_root: Path) -> pd.DataFrame:
    rows = []
    for run_dir in simulation_run_dirs(raw_root, 'used_config.yml'):
        clock_path = run_dir / 'wall_clock.csv'
        if not clock_path.is_file():
            continue
        clock = pd.read_csv(clock_path)
        seconds = pd.to_numeric(clock.get('post_score_s'), errors='coerce').sum(min_count=1)
        context = run_context(run_dir, raw_root)
        rows.append({**context, 'score_time_s': seconds})
    return pd.DataFrame(rows)


if RUN_TABLE_3:
    root = claim_root('Table3')
    accuracy = read_simulation_accuracy(root)
    curves = run_accuracy_curves(accuracy)
    result_rows = []
    group_keys = ['run_id', 'task', 'importance_metric']
    for keys, frame in curves.groupby(group_keys, dropna=False):
        frame = frame.sort_values('round')
        final_accuracy = float(frame['accuracy'].iloc[-1])
        auc_thousands = float(np.trapz(frame['accuracy'], frame['round']) / 1000.0)
        result_rows.append({
            'run_id': keys[0], 'task': keys[1], 'importance_metric': keys[2],
            'final_accuracy': final_accuracy, 'auc_1e3': auc_thousands,
        })
    per_run = pd.DataFrame(result_rows)
    times = read_score_times(root)
    if not times.empty:
        per_run = per_run.merge(times[['run_id', 'score_time_s']], on='run_id', how='left')
    else:
        per_run['score_time_s'] = np.nan
    summary = per_run.groupby(['task', 'importance_metric']).agg(
        runs=('run_id', 'nunique'),
        final_accuracy_mean=('final_accuracy', 'mean'),
        final_accuracy_sd=('final_accuracy', 'std'),
        auc_1e3_mean=('auc_1e3', 'mean'),
        auc_1e3_sd=('auc_1e3', 'std'),
        score_time_s_mean=('score_time_s', 'mean'),
    ).reset_index()
    baseline = summary[summary['importance_metric'] == 'Parameter magnitude'][
        ['task', 'score_time_s_mean']
    ].rename(columns={'score_time_s_mean': 'baseline_time_s'})
    summary = summary.merge(baseline, on='task', how='left')
    summary['scoring_cost_relative'] = summary['score_time_s_mean'] / summary['baseline_time_s']
    summary = summary.drop(columns='baseline_time_s').sort_values(['task', 'importance_metric'])
    display(summary.round(3))
    if EXPORT_OUTPUTS:
        summary.to_csv(output_root('Table3') / 'table3_summary.csv', index=False)
else:
    print('Table 3: disabled')

## Figures 19 and 20 — communication

In [ ]:
def read_packets(raw_root: Path) -> pd.DataFrame:
    frames = []
    for packet_path in raw_root.rglob('communication_packets.csv'):
        run_dir = packet_path.parent
        if not (run_dir / 'used_config.yml').is_file():
            continue
        frame = pd.read_csv(packet_path)
        if frame.empty:
            continue
        for column in ('global_round', 'total_bytes'):
            if column in frame:
                frame[column] = pd.to_numeric(frame[column], errors='coerce')
        for column in ('source_device', 'destination_device'):
            if column in frame:
                device_index = frame[column].astype('string').str.extract(r'^device_(\d+)$', expand=False)
                frame[column] = pd.to_numeric(device_index, errors='coerce').astype('Int64')
        context = run_context(run_dir, raw_root)
        for key, value in context.items():
            frame[key] = value
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f'No communication_packets.csv runs found under {raw_root}')
    return pd.concat(frames, ignore_index=True)


from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

PAPER_COMM_METHOD_ORDER = ['tinyGist', 'DFA', 'SDFA', 'SP']
PAPER_COMM_COLORS = {
    'DFA': '#1C39BB',
    'tinyGist': '#870074',
    'SDFA': '#00A693',
    'SP': '#F38400',
}


def paper_method_colormap(method: str) -> LinearSegmentedColormap:
    color = PAPER_COMM_COLORS[method]
    rgb = np.array([int(color[i:i + 2], 16) / 255 for i in (1, 3, 5)])
    ramp = np.linspace(0, 1, 256) ** 1.5
    colors = 1 - ramp[:, None] * (1 - rgb[None, :])
    return LinearSegmentedColormap.from_list(f'{method}_paper', colors)


def require_condition_grid(actual, expected, label: str) -> None:
    if not STRICT_INPUT:
        return
    missing = sorted(set(expected) - set(actual), key=str)
    unexpected = sorted(set(actual) - set(expected), key=str)
    if missing or unexpected:
        raise ValueError(f'{label}: missing={missing}, unexpected={unexpected}')


def paper_nonzero_tick(maximum: float) -> float:
    target = maximum * 0.8
    if target <= 0:
        return 1.0
    exponent = math.floor(math.log10(target))
    candidates = [
        factor * (10 ** power)
        for power in range(exponent - 1, exponent + 2)
        for factor in (1, 2, 2.5, 5, 10)
    ]
    candidates = [value for value in candidates if 0 < value <= maximum * 1.02]
    return min(candidates, key=lambda value: abs(value - target)) if candidates else maximum


def compact_tick_label(value: float, _position=None) -> str:
    if math.isclose(value, 0.0, abs_tol=1e-15):
        return '0'
    label = f'{value:g}' if abs(value) >= 1 else f'{value:.3g}'
    return label[1:] if label.startswith('0.') else label


if RUN_FIGURE_19:
    root = claim_root('Figure19')
    packets = read_packets(root)
    data = packets[(packets['panel_id'] == FIGURE_19_PANEL_ID) &
                   (packets['method'].isin(PAPER_COMM_METHOD_ORDER)) &
                   (packets['status'] == 'delivered')].copy()
    if data.empty:
        raise ValueError(f'No delivered Figure 19 packets for panel {FIGURE_19_PANEL_ID}')
    require_conditions(data['method'], PAPER_COMM_METHOD_ORDER, 'Figure 19 methods')
    devices = sorted(set(data['source_device'].dropna().astype(int)) |
                     set(data['destination_device'].dropna().astype(int)))
    if STRICT_INPUT and devices != list(range(10)):
        raise ValueError(f'Figure 19 expects devices 0..9, found: {devices}')

    fig, axes = plt.subplots(2, 2, figsize=(3.19, 3.36), dpi=200, squeeze=False)
    matrix_records = []
    for panel_index, (axis, method) in enumerate(zip(axes.flat, PAPER_COMM_METHOD_ORDER)):
        method_data = data[data['method'] == method]
        per_run = method_data.groupby(
            ['run_id', 'source_device', 'destination_device'], as_index=False
        )['total_bytes'].sum()
        mean = (
            per_run.groupby(['source_device', 'destination_device'])['total_bytes'].mean()
            / (1024 ** 2)
        )
        matrix = mean.unstack(fill_value=0).reindex(
            index=devices, columns=devices, fill_value=0
        )
        image = axis.imshow(matrix, cmap=paper_method_colormap(method), aspect='equal', vmin=0)
        axis.xaxis.tick_top()
        axis.xaxis.set_label_position('top')
        axis.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False,
                         labelsize=4.4, length=2, pad=1)
        axis.set_xlabel('destination', fontsize=5.2, labelpad=1)
        axis.set_ylabel('source', fontsize=5.2, labelpad=1)
        axis.set_xticks(range(len(devices)), devices)
        axis.set_yticks(range(len(devices)), devices)
        for row in range(len(devices)):
            for column in range(len(devices)):
                value = float(matrix.iloc[row, column])
                matrix_records.append({
                    'panel_id': FIGURE_19_PANEL_ID,
                    'task': str(method_data['task'].iloc[0]),
                    'method': method,
                    'source_device': devices[row],
                    'destination_device': devices[column],
                    'mean_MB': value,
                })
                if value > 0:
                    label = f'{value:.3g}'
                    if label.startswith('0.'):
                        label = label[1:]
                    axis.text(column, row, label, ha='center', va='center',
                              color='black', fontsize=2.6)
        for spine in axis.spines.values():
            spine.set_linewidth(0.55)
        colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.035)
        colorbar.ax.tick_params(labelsize=4.2, length=2, pad=1)
        colorbar.outline.set_linewidth(0.55)
        axis.text(0.5, -0.205, f'({chr(97 + panel_index)}) {method}',
                  transform=axis.transAxes, ha='center', va='top',
                  fontsize=6.6, fontfamily='serif', fontweight='bold',
                  color=PAPER_COMM_COLORS[method] if method == 'tinyGist' else 'black')
    fig.subplots_adjust(left=0.08, right=0.90, bottom=0.095, top=0.91,
                        wspace=0.42, hspace=0.62)
    if EXPORT_OUTPUTS:
        pd.DataFrame(matrix_records).to_csv(
            output_root('Figure19') / 'figure19_peer_matrix.csv', index=False
        )
    save_figure(fig, 'Figure19', 'figure19_peer_matrix', bbox_inches=None)
    display_figure(fig)
else:
    print('Figure 19: disabled')


if RUN_FIGURE_20:
    root = claim_root('Figure20')
    packets = read_packets(root)
    packets = packets[packets['method'].isin(COMM_METHOD_ORDER)].copy()
    packet_devices = sorted(set(packets['source_device'].dropna().astype(int)) |
                            set(packets['destination_device'].dropna().astype(int)))
    if STRICT_INPUT and packet_devices != list(range(10)):
        raise ValueError(f'Figure 20 expects devices 0..9, found: {packet_devices}')
    run_keys = ['run_id', 'panel_id', 'task', 'method']
    run_grid = packets[run_keys].drop_duplicates()
    tx = (
        packets[packets['source_device'].eq(FIGURE_20_DEVICE_INDEX)]
        .groupby(run_keys, as_index=False)['total_bytes'].sum()
        .rename(columns={'total_bytes': 'tx_bytes'})
    )
    rx = (
        packets[packets['destination_device'].eq(FIGURE_20_DEVICE_INDEX) &
                packets['status'].eq('delivered')]
        .groupby(run_keys, as_index=False)['total_bytes'].sum()
        .rename(columns={'total_bytes': 'rx_bytes'})
    )
    per_run = run_grid.merge(tx, on=run_keys, how='left').merge(rx, on=run_keys, how='left')
    if STRICT_INPUT:
        missing_tx = per_run.loc[per_run['tx_bytes'].isna(), run_keys].to_dict('records')
        missing_rx = per_run.loc[per_run['rx_bytes'].isna(), run_keys].to_dict('records')
        if missing_tx or missing_rx:
            raise ValueError(
                f'Figure 20 device_{FIGURE_20_DEVICE_INDEX} direction data missing: '
                f'TX={missing_tx}, RX={missing_rx}'
            )
    per_run[['tx_bytes', 'rx_bytes']] = per_run[['tx_bytes', 'rx_bytes']].fillna(0)
    summary = (
        per_run.groupby(['panel_id', 'task', 'method'])[['tx_bytes', 'rx_bytes']]
        .mean().reset_index()
    )
    summary['tx_MB'] = summary['tx_bytes'] / (1024 ** 2)
    summary['rx_MB'] = summary['rx_bytes'] / (1024 ** 2)
    summary['device'] = f'device_{FIGURE_20_DEVICE_INDEX}'

    panels = sorted(summary['panel_id'].dropna().astype(int).unique())
    require_conditions(panels, range(1, 16), 'Figure 20 panels')
    expected_grid = {(panel, method) for panel in range(1, 16) for method in COMM_METHOD_ORDER}
    actual_grid = {
        (int(panel), method) for panel, method in
        summary[['panel_id', 'method']].itertuples(index=False, name=None)
    }
    require_condition_grid(actual_grid, expected_grid, 'Figure 20 panel/method grid')

    fig = plt.figure(figsize=(3.32, 1.83), dpi=220)
    grid = fig.add_gridspec(2, 16, left=0.02, right=0.995, bottom=0.14, top=0.76,
                            wspace=0.72, hspace=1.18)
    axes = []
    for index in range(8):
        axes.append(fig.add_subplot(grid[0, 2 * index:2 * index + 2]))
    for index in range(7):
        axes.append(fig.add_subplot(grid[1, 1 + 2 * index:1 + 2 * index + 2]))

    method_y = np.arange(len(COMM_METHOD_ORDER))
    bar_height = 0.26
    for axis, panel in zip(axes, panels):
        frame = summary[summary['panel_id'] == panel].set_index('method').reindex(COMM_METHOD_ORDER)
        for y, method in zip(method_y, COMM_METHOD_ORDER):
            color = PAPER_COMM_COLORS[method]
            tx_value = float(frame.loc[method, 'tx_MB'])
            rx_value = float(frame.loc[method, 'rx_MB'])
            axis.barh(y - bar_height / 2, tx_value, height=bar_height,
                      color=color, edgecolor='black', linewidth=0.45, hatch='...', alpha=0.92, zorder=3)
            axis.barh(y + bar_height / 2, rx_value, height=bar_height,
                      color=color, edgecolor='black', linewidth=0.45, hatch='xxx', alpha=0.92, zorder=3)
        maximum = float(frame[['tx_MB', 'rx_MB']].to_numpy().max())
        axis.set_xlim(0, maximum * 1.08 if maximum > 0 else 1)
        axis.set_ylim(-0.55, len(COMM_METHOD_ORDER) - 0.45)
        axis.set_yticks([])
        axis.set_xticks([0, paper_nonzero_tick(maximum)])
        axis.xaxis.set_major_formatter(FuncFormatter(compact_tick_label))
        axis.tick_params(axis='x', labelsize=4.6, length=2, width=0.5, pad=1)
        tick_labels = axis.get_xticklabels()
        tick_labels[0].set_ha('left')
        tick_labels[-1].set_ha('right')
        axis.grid(axis='x', color='#d0d0d0', alpha=0.5, linewidth=0.4, zorder=0)
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
        axis.spines['left'].set_linewidth(0.55)
        axis.spines['bottom'].set_linewidth(0.55)
        axis.text(0.5, -0.39, f'({chr(96 + panel)})', transform=axis.transAxes,
                  ha='center', va='top', fontsize=7.0, fontfamily='serif', fontweight='bold')

    legend_handles = [
        *[Patch(facecolor=PAPER_COMM_COLORS[method], edgecolor='black', linewidth=0.45, label=method)
          for method in COMM_METHOD_ORDER],
        Patch(facecolor='white', edgecolor='black', linewidth=0.45, hatch='...', label='TX'),
        Patch(facecolor='white', edgecolor='black', linewidth=0.45, hatch='xxx', label='RX'),
    ]
    fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.985),
               ncol=6, frameon=False, fontsize=5.1, handlelength=1.15,
               handleheight=0.85, columnspacing=0.55, handletextpad=0.18, borderpad=0)
    if EXPORT_OUTPUTS:
        summary[['panel_id', 'task', 'method', 'device', 'tx_MB', 'rx_MB']].to_csv(
            output_root('Figure20') / 'figure20_data_volume.csv', index=False
        )
    save_figure(fig, 'Figure20', 'figure20_data_volume', bbox_inches=None)
    display_figure(fig)
else:
    print('Figure 20: disabled')

## Figure 21 — Spearman relationship

In [ ]:
def read_importance(raw_root: Path) -> pd.DataFrame:
    frames = []
    for csv_path in raw_root.rglob('importance_correlation.csv'):
        run_dir = csv_path.parent
        if not (run_dir / 'used_config.yml').is_file():
            continue
        frame = pd.read_csv(csv_path)
        if frame.empty or 'spearman' not in frame:
            continue
        for column in ('round', 'device', 'spearman'):
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
        context = run_context(run_dir, raw_root)
        for key, value in context.items():
            frame[key] = value
        frame['reference_label'] = frame['reference_score'].map(
            lambda value: normalize_importance_metric(value)
        )
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f'No importance_correlation.csv runs found under {raw_root}')
    return pd.concat(frames, ignore_index=True)


if RUN_FIGURE_21:
    root = claim_root('Figure21')
    data = read_importance(root).dropna(subset=['spearman'])
    per_run = data.groupby(['run_id', 'task', 'reference_label', 'round'], as_index=False)['spearman'].mean()
    summary = per_run.groupby(['task', 'reference_label', 'round'])['spearman'].agg(
        mean='mean', sd='std', runs='count'
    ).reset_index()
    references = list(REFERENCE_COLORS)
    require_conditions(summary['reference_label'], references, 'Figure 21 reference metrics')
    tasks = sorted(summary['task'].unique())
    if STRICT_INPUT and len(tasks) != 2:
        raise ValueError(f'Figure 21 expects two tasks, found {tasks}')
    fig, axes = plt.subplots(1, len(tasks), figsize=(9.2, 3.3), squeeze=False, sharey=True)
    for axis, task in zip(axes.flat, tasks):
        for reference in references:
            line = summary[(summary['task'] == task) &
                           (summary['reference_label'] == reference)].sort_values('round')
            if line.empty:
                continue
            axis.plot(line['round'], line['mean'], label=reference,
                      color=REFERENCE_COLORS[reference], lw=1.3)
            sd = line['sd'].fillna(0)
            axis.fill_between(line['round'], line['mean'] - sd, line['mean'] + sd,
                              color=REFERENCE_COLORS[reference], alpha=0.12)
        axis.set_title(task)
        axis.set_xlabel('Round')
        axis.grid(alpha=0.25)
    axes[0, 0].set_ylabel('Spearman rho')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.78))
    save_figure(fig, 'Figure21', 'figure21_spearman')
    display_figure(fig)
else:
    print('Figure 21: disabled')

## Figure 22 — DP-SGD privacy budgets

For a two-round `--quick-check`, this section validates the result-to-plot pipeline only. The configured 200-round target ε labels each condition; the two-round planned and accounted ε values are retained only as provenance and are not expected to match the target. Accuracy and ε values from this smoke test are not a numerical reproduction of the paper.

In [ ]:
if RUN_FIGURE_22:
    root = claim_root('Figure22')
    accuracy = read_simulation_accuracy(root)
    accuracy = accuracy[np.isclose(accuracy['delta'].astype(float), 1e-5, equal_nan=False)]
    if accuracy.empty:
        raise ValueError('Figure 22 requires delta=1e-5 DP-SGD runs')
    expected_epsilons = [0.5, 1, 2, 4, 8, 20]
    provenance_columns = [
        'run_id', 'source_stem', 'task', 'configured_target_epsilon',
        'planned_epsilon', 'actual_epsilon', 'delta',
    ]
    provenance = accuracy[provenance_columns].drop_duplicates().sort_values(
        ['task', 'configured_target_epsilon', 'run_id']
    )
    tasks = sorted(provenance['task'].unique())
    if STRICT_INPUT:
        if len(tasks) != 2:
            raise ValueError(f'Figure 22 expects two tasks, found {tasks}')
        expected_grid = {(task, epsilon) for task in tasks for epsilon in expected_epsilons}
        observed_grid = set(zip(provenance['task'], provenance['configured_target_epsilon']))
        if observed_grid != expected_grid or len(provenance) != len(expected_grid):
            raise ValueError(
                'Figure 22 task/target-epsilon grid mismatch: '
                f'missing={sorted(expected_grid - observed_grid)}, '
                f'extra={sorted(observed_grid - expected_grid)}, runs={len(provenance)}'
            )
        for column in ('planned_epsilon', 'actual_epsilon'):
            values = pd.to_numeric(provenance[column], errors='coerce')
            if not np.isfinite(values).all() or (values <= 0).any():
                raise ValueError(f'Figure 22 {column} provenance must be finite and positive')
    summary = curve_summary(accuracy, ['task', 'configured_target_epsilon'])
    cmap = plt.get_cmap('viridis')
    epsilon_colors = {value: cmap(index / (len(expected_epsilons) - 1))
                      for index, value in enumerate(expected_epsilons)}
    fig, axes = plt.subplots(1, len(tasks), figsize=(9.2, 3.3), squeeze=False, sharey=True)
    for axis, task in zip(axes.flat, tasks):
        for epsilon in expected_epsilons:
            line = summary[(summary['task'] == task) &
                           np.isclose(summary['configured_target_epsilon'], epsilon)].sort_values('round')
            if line.empty:
                continue
            axis.plot(line['round'], line['mean'], color=epsilon_colors[epsilon],
                      label=f'target epsilon={epsilon:g}', lw=1.4)
            sd = line['sd'].fillna(0)
            axis.fill_between(line['round'], line['mean'] - sd, line['mean'] + sd,
                              color=epsilon_colors[epsilon], alpha=0.1)
        axis.set_title(task)
        axis.set_xlabel('Round')
        axis.grid(alpha=0.25)
    axes[0, 0].set_ylabel('Accuracy (%)')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.86))
    save_figure(fig, 'Figure22', 'figure22_dp_sgd')
    if EXPORT_OUTPUTS:
        provenance.to_csv(
            output_root('Figure22') / 'figure22_privacy_provenance.csv', index=False
        )
    display(provenance)
    display_figure(fig)
else:
    print('Figure 22: disabled')

## Shared firmware-log reader

In [ ]:
ANSI_ESCAPE = re.compile(r'\x1b\[[0-?]*[ -/]*[@-~]')
SUCCESS_RATE = re.compile(
    r'\[ROUND\s+(\d+)\].*?###\s+Success Rate:\s*'
    r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
)
DEVICE_ID = re.compile(r"This Device's ID:\s*(\d+)")
CLUSTER = re.compile(r'(\d+)H[_-]?(\d+)M[_-]?(\d+)L', re.IGNORECASE)


def infer_firmware_context(path: Path, root: Path, environment: str) -> dict:
    relative = path.relative_to(root)
    text = '/'.join(relative.parts)
    lowered = text.lower()
    task = 'CNN–Gesture' if 'gesture' in lowered else 'FCN–MNIST' if 'mnist' in lowered else ''
    method = ''
    for candidate in ('gist_ada', 'tinygist', 'sdfa', 'dfa'):
        if candidate in lowered:
            method = normalize_method(candidate)
            break
    count_match = re.search(r'(\d+)[_-]?devices', lowered)
    if count_match is None:
        count_match = re.search(r'/(5|10)/', f'/{lowered}/')
    client_count = int(count_match.group(1)) if count_match else np.nan
    cluster_match = CLUSTER.search(text)
    cluster = ''
    if cluster_match:
        cluster = f'{cluster_match.group(1)}H/{cluster_match.group(2)}M/{cluster_match.group(3)}L'
    tier = ''
    for token, label in (('high', 'High'), ('medium', 'Mid'), ('mid', 'Mid'), ('low', 'Low')):
        if re.search(fr'(^|[/_.-]){token}([/_.-]|$)', lowered):
            tier = label
            break
    return {
        'environment': environment,
        'task': task,
        'method': method,
        'client_count': client_count,
        'cluster': cluster,
        'device_tier': tier,
        'run_id': str(relative.parent),
        'source_file': str(relative),
    }


def manifest_records(root: Path) -> dict[Path, dict]:
    records = {}
    for manifest_path in root.rglob('manifest.csv'):
        manifest = pd.read_csv(manifest_path)
        if 'path' not in manifest:
            continue
        for row in manifest.to_dict('records'):
            path = Path(str(row['path']))
            if not path.is_absolute():
                path = (manifest_path.parent / path).resolve()
            records[path] = row
    return records


def read_firmware_logs(root: Path, environment: str) -> pd.DataFrame:
    manifest = manifest_records(root)
    rows = []
    for log_path in sorted(root.rglob('*.log')):
        context = infer_firmware_context(log_path, root, environment)
        manifest_row = manifest.get(log_path.resolve(), {})
        context.update({key: value for key, value in manifest_row.items() if not pd.isna(value)})
        context['method'] = normalize_method(context.get('method', ''))
        task_token = str(context.get('task', '')).lower()
        if 'gesture' in task_token:
            context['task'] = 'CNN–Gesture'
        elif 'mnist' in task_token:
            context['task'] = 'FCN–MNIST'
        tier_token = str(context.get('device_tier', '')).lower()
        context['device_tier'] = {'high': 'High', 'medium': 'Mid', 'mid': 'Mid', 'low': 'Low'}.get(
            tier_token, context.get('device_tier', '')
        )
        device_id = context.get('device_id', np.nan)
        with log_path.open('r', encoding='utf-8', errors='replace') as handle:
            for raw_line in handle:
                line = ANSI_ESCAPE.sub('', raw_line)
                id_match = DEVICE_ID.search(line)
                if id_match:
                    device_id = int(id_match.group(1))
                match = SUCCESS_RATE.search(line)
                if not match:
                    continue
                value = float(match.group(2))
                if abs(value) <= 1.5:
                    value *= 100.0
                rows.append({
                    **context,
                    'device_id': device_id,
                    'raw_round': int(match.group(1)),
                    'round': int(match.group(1)) + 1,
                    'accuracy': value,
                })
    if not rows:
        raise FileNotFoundError(f'No firmware Success Rate records found under {root}')
    return pd.DataFrame(rows)

## Figure 14 — fidelity across environments

Set `FIGURE_14_PATH` near the top to one of:

- `emulation_only`: Emulation only.
- `emulation_simulation`: compare Emulation with Simulation.
- `emulation_simulation_real`: compare Emulation, Simulation, and homogeneous real ESP32-S3 devices.

With `STRICT_INPUT=True`, every selected environment must contain both tasks, DFA/SDFA/tinyGist, 5- and 10-device groups, exactly 5 or 10 unique devices per run, and the first 200 report rounds. The full path additionally validates ESP32-S3, 240 MHz, and ESP-IDF 5.2.0 from each real-device manifest.

In [ ]:
if RUN_FIGURE_14:
    path_environments = {
        'emulation_only': ('Emulation',),
        'emulation_simulation': ('Simulation', 'Emulation'),
        'emulation_simulation_real': ('Simulation', 'Emulation', 'Real'),
    }
    if FIGURE_14_PATH not in path_environments:
        raise ValueError(
            f'Unknown FIGURE_14_PATH={FIGURE_14_PATH!r}; choose one of {sorted(path_environments)}'
        )
    selected_environments = path_environments[FIGURE_14_PATH]
    conditions = [
        ('Gesture', 'tinyGist', 5), ('Gesture', 'SDFA', 5), ('Gesture', 'DFA', 5),
        ('MNIST', 'tinyGist', 5), ('MNIST', 'SDFA', 5), ('MNIST', 'DFA', 5),
        ('Gesture', 'tinyGist', 10), ('Gesture', 'SDFA', 10), ('Gesture', 'DFA', 10),
        ('MNIST', 'tinyGist', 10), ('MNIST', 'SDFA', 10), ('MNIST', 'DFA', 10),
    ]
    condition_set = set(conditions)
    combined_frames = []
    if 'Simulation' in selected_environments:
        simulation = read_simulation_accuracy(claim_root('Figure14', 'simulation'))
        simulation = simulation[
            ['run_id', 'task', 'method', 'client_count', 'round', 'device', 'accuracy']
        ].copy()
        # Simulation tables label completed FL rounds as 1..200. Figure 14
        # reports the corresponding zero-based positions 0..199, like the
        # firmware parser's raw round numbers and the paper reference arrays.
        simulation['round'] = pd.to_numeric(simulation['round'], errors='coerce') - 1
        simulation['environment'] = 'Simulation'
        simulation['run_id'] = 'simulation/' + simulation['run_id'].astype(str)
        simulation['task'] = simulation['task'].replace(
            {'CNN–Gesture': 'Gesture', 'FCN–MNIST': 'MNIST'}
        )
        combined_frames.append(simulation)
    for environment, directory in (('Emulation', 'emulation'), ('Real', 'real')):
        if environment not in selected_environments:
            continue
        firmware = read_firmware_logs(claim_root('Figure14', directory), environment)
        firmware['task'] = firmware['task'].replace(
            {'CNN–Gesture': 'Gesture', 'FCN–MNIST': 'MNIST'}
        )
        if environment == 'Real' and STRICT_INPUT:
            required_hardware = ['board', 'target', 'cpu_mhz', 'idf_version']
            missing_hardware = [column for column in required_hardware if column not in firmware]
            if missing_hardware:
                raise ValueError(f'Figure 14 real manifests are missing fields: {missing_hardware}')
            board = (firmware['board'].astype(str).str.lower()
                     .str.replace(r'[\s_-]+', '', regex=True))
            target = firmware['target'].astype(str).str.lower().str.strip()
            cpu_mhz = pd.to_numeric(firmware['cpu_mhz'], errors='coerce')
            idf_version = firmware['idf_version'].astype(str).str.strip()
            invalid_hardware = ~(
                board.eq('esp32s3')
                & target.eq('esp32s3')
                & cpu_mhz.eq(240)
                & idf_version.eq('5.2.0')
            )
            if invalid_hardware.any():
                invalid_sources = sorted(firmware.loc[invalid_hardware, 'source_file'].astype(str).unique())
                raise ValueError(
                    'Figure 14 real manifests must identify ESP32-S3/esp32s3/240 MHz/'
                    f'ESP-IDF 5.2.0 for every log; invalid files: {invalid_sources}'
                )
        firmware['client_count'] = pd.to_numeric(firmware['client_count'], errors='coerce')
        firmware['device_id'] = pd.to_numeric(firmware['device_id'], errors='coerce')
        if firmware['client_count'].isna().any():
            raise ValueError(f'Figure 14 {environment} logs contain an unknown client count')
        firmware['client_count'] = firmware['client_count'].astype(int)
        identity_keys = ['run_id', 'task', 'method', 'client_count']
        source_identity = firmware.groupby(
            [*identity_keys, 'source_file'], dropna=False
        ).agg(
            records=('device_id', 'size'),
            identified_records=('device_id', 'count'),
            unique_device_ids=('device_id', 'nunique'),
        ).reset_index()
        malformed_sources = source_identity[
            (source_identity['identified_records'] != source_identity['records'])
            | (source_identity['unique_device_ids'] != 1)
        ]
        if not malformed_sources.empty:
            raise ValueError(
                f'Each Figure 14 {environment} log must contain exactly one device ID:\n'
                + malformed_sources.to_string(index=False)
            )
        firmware_run_identity = firmware.groupby(identity_keys, dropna=False).agg(
            source_files=('source_file', 'nunique'),
            unique_device_ids=('device_id', 'nunique'),
        ).reset_index()
        malformed_firmware_runs = firmware_run_identity[
            (firmware_run_identity['source_files'] != firmware_run_identity['client_count'])
            | (firmware_run_identity['unique_device_ids'] != firmware_run_identity['client_count'])
        ]
        if not malformed_firmware_runs.empty:
            raise ValueError(
                f'Figure 14 {environment} runs require one source log per declared device:\n'
                + malformed_firmware_runs.to_string(index=False)
            )
        expected_id_start = 10 if environment == 'Emulation' else 40
        invalid_id_sets = []
        for run_key, run_frame in firmware.groupby(identity_keys, dropna=False):
            client_count = int(run_key[-1])
            actual_ids = set(run_frame['device_id'].dropna().astype(int))
            expected_ids = set(range(expected_id_start, expected_id_start + client_count))
            if actual_ids != expected_ids:
                invalid_id_sets.append({
                    **dict(zip(identity_keys, run_key)),
                    'actual_ids': sorted(actual_ids),
                    'expected_ids': sorted(expected_ids),
                })
        if invalid_id_sets:
            raise ValueError(
                f'Figure 14 {environment} runs use the wrong firmware device-ID set:\n'
                + pd.DataFrame(invalid_id_sets).to_string(index=False)
            )
        firmware['round'] = firmware['raw_round']
        firmware = firmware.rename(columns={'device_id': 'device'})
        combined_frames.append(
            firmware[
                ['run_id', 'task', 'method', 'client_count', 'round', 'device', 'accuracy', 'environment']
            ].copy()
        )
    combined = pd.concat(combined_frames, ignore_index=True)
    combined['client_count'] = pd.to_numeric(combined['client_count'], errors='coerce')
    combined['round'] = pd.to_numeric(combined['round'], errors='coerce')
    combined = combined.dropna(subset=['client_count', 'round', 'device', 'accuracy']).copy()
    combined['client_count'] = combined['client_count'].astype(int)
    combined['round'] = combined['round'].astype(int)
    combined['condition'] = list(zip(combined['task'], combined['method'], combined['client_count']))
    combined = combined[combined['condition'].isin(condition_set)].drop(columns='condition')
    combined = combined[
        combined['round'].between(0, FIGURE_14_REPORT_ROUNDS - 1, inclusive='both')
    ].copy()
    if combined.empty:
        raise ValueError(f'Figure 14 path {FIGURE_14_PATH!r} has no usable records')
    if STRICT_INPUT:
        expected = {
            (*condition, environment)
            for condition in conditions
            for environment in selected_environments
        }
        actual = set(zip(
            combined['task'], combined['method'], combined['client_count'], combined['environment']
        ))
        missing = sorted(expected - actual)
        if missing:
            raise ValueError(f'Figure 14 path {FIGURE_14_PATH!r} is missing conditions: {missing}')
    run_coverage = combined.groupby(
        ['environment', 'run_id', 'task', 'method', 'client_count'], dropna=False
    ).agg(
        unique_devices=('device', 'nunique'),
        records=('accuracy', 'size'),
        minimum_round=('round', 'min'),
        maximum_round=('round', 'max'),
    ).reset_index()
    malformed_runs = run_coverage[run_coverage['unique_devices'] != run_coverage['client_count']]
    if not malformed_runs.empty:
        raise ValueError(
            'Figure 14 runs must contain exactly client_count unique devices:\n'
            + malformed_runs.to_string(index=False)
        )
    device_coverage = combined.groupby(
        ['environment', 'run_id', 'task', 'method', 'client_count', 'device'], dropna=False
    ).agg(
        records=('accuracy', 'size'),
        recorded_rounds=('round', 'nunique'),
        minimum_round=('round', 'min'),
        maximum_round=('round', 'max'),
    ).reset_index()
    if STRICT_INPUT:
        incomplete_devices = device_coverage[
            (device_coverage['records'] != FIGURE_14_REPORT_ROUNDS)
            | (device_coverage['recorded_rounds'] != FIGURE_14_REPORT_ROUNDS)
            | (device_coverage['minimum_round'] != 0)
            | (device_coverage['maximum_round'] != FIGURE_14_REPORT_ROUNDS - 1)
        ]
        if not incomplete_devices.empty:
            raise ValueError(
                f'Figure 14 requires rounds 0..{FIGURE_14_REPORT_ROUNDS - 1} for every device:\n'
                + incomplete_devices.to_string(index=False)
            )
    summary = combined.groupby(
        ['task', 'method', 'client_count', 'environment', 'round']
    )['accuracy'].agg(
        mean='mean', sd=lambda values: values.std(ddof=0), samples='count'
    ).reset_index()
    env_colors = {'Simulation': '#1C39BB', 'Emulation': '#317873', 'Real': '#882D17'}
    env_markers = {'Simulation': 'o', 'Emulation': 's', 'Real': 'D'}
    fig, axes = plt.subplots(3, 4, figsize=(12.5, 7.2), sharex=True, sharey=True)
    for panel_index, (axis, (task, method, clients)) in enumerate(zip(axes.flat, conditions)):
        frame = summary[(summary['task'] == task) & (summary['method'] == method) &
                        (summary['client_count'] == clients)]
        for environment in selected_environments:
            line = frame[frame['environment'] == environment].sort_values('round')
            if line.empty:
                continue
            mark_every = max(1, len(line) // 10)
            axis.plot(
                line['round'], line['mean'], color=env_colors[environment],
                marker=env_markers[environment], markevery=mark_every, markersize=2.5,
                label=environment, lw=2.0,
            )
            sd = line['sd'].fillna(0)
            axis.fill_between(line['round'], line['mean'] - sd, line['mean'] + sd,
                              color=env_colors[environment], alpha=0.15)
        axis.set_title(f'({chr(ord("a") + panel_index)}) {task}/{method}/{clients}')
        axis.set_xlim(0, FIGURE_14_REPORT_ROUNDS + 1)
        axis.set_xticks([0, 100, 200])
        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)
        axis.grid(alpha=0.3)
    for axis in axes[-1, :]:
        axis.set_xlabel('Round')
    for axis in axes[:, 0]:
        axis.set_ylabel('Accuracy (%)')
    handles_by_label = {}
    for axis in axes.flat:
        for handle, label in zip(*axis.get_legend_handles_labels()):
            handles_by_label.setdefault(label, handle)
    legend_labels = [environment for environment in selected_environments if environment in handles_by_label]
    fig.legend(
        [handles_by_label[label] for label in legend_labels], legend_labels,
        loc='upper center', ncol=len(legend_labels), frameon=False,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    output_stems = {
        'emulation_only': 'figure14_emulation',
        'emulation_simulation': 'figure14_emulation_simulation',
        'emulation_simulation_real': 'figure14_emulation_simulation_real',
    }
    output_stem = output_stems[FIGURE_14_PATH]
    if EXPORT_OUTPUTS:
        target = output_root('Figure14')
        summary.to_csv(target / f'{output_stem}_summary.csv', index=False)
        run_coverage.to_csv(target / f'{output_stem}_coverage.csv', index=False)
    save_figure(fig, 'Figure14', output_stem)
    display_figure(fig)
else:
    print('Figure 14: disabled')

## Figure 16 — heterogeneous physical-device clusters

For this section, place a `manifest.csv` beside the logs. At minimum it should map each log `path` to `task`, `method`, `client_count`, `cluster`, and `device_tier` (`High`, `Mid`, or `Low`).

In [ ]:
if RUN_FIGURE_16:
    root = claim_root('Figure16', 'real')
    data = read_firmware_logs(root, 'Real')
    required_columns = ['task', 'method', 'cluster', 'device_tier']
    missing_metadata = [column for column in required_columns
                        if column not in data or data[column].astype(str).str.len().eq(0).all()]
    if missing_metadata:
        raise ValueError(f'Figure 16 manifest is missing usable fields: {missing_metadata}')
    round_targets = [25, 50, 200]
    data = data[data['round'].isin(round_targets)].copy()
    if data.empty:
        raise ValueError(f'Figure 16 requires rounds {round_targets}')
    require_conditions(data['method'], ['tinyGist', 'SDFA', 'DFA'], 'Figure 16 methods')
    require_conditions(data['device_tier'], ['High', 'Mid', 'Low'], 'Figure 16 device tiers')
    per_run = data.groupby(
        ['run_id', 'task', 'cluster', 'method', 'device_tier', 'round'], as_index=False
    )['accuracy'].mean()
    summary = per_run.groupby(
        ['task', 'cluster', 'method', 'device_tier', 'round']
    )['accuracy'].agg(mean='mean', sd='std').reset_index()
    panels = sorted(set(zip(summary['task'], summary['cluster'])))
    if STRICT_INPUT and len(panels) != 4:
        raise ValueError(f'Figure 16 expects four task/cluster panels, found {panels}')
    tier_hatches = {'High': 'xx', 'Mid': '//', 'Low': '..'}
    methods = ['tinyGist', 'SDFA', 'DFA']
    tiers = ['High', 'Mid', 'Low']
    fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.2), squeeze=False, sharey=True)
    width = 0.08
    for axis, (task, cluster) in zip(axes.flat, panels):
        frame = summary[(summary['task'] == task) & (summary['cluster'] == cluster)]
        x = np.arange(len(round_targets))
        for method_index, method in enumerate(methods):
            for tier_index, tier in enumerate(tiers):
                offset_index = method_index * len(tiers) + tier_index - 4
                values = []
                errors = []
                for round_value in round_targets:
                    row = frame[(frame['method'] == method) &
                                (frame['device_tier'] == tier) &
                                (frame['round'] == round_value)]
                    values.append(row['mean'].iloc[0] if not row.empty else np.nan)
                    errors.append(row['sd'].fillna(0).iloc[0] if not row.empty else 0)
                axis.bar(x + offset_index * width, values, width=width,
                         yerr=errors, capsize=1.5, color=METHOD_COLORS[method],
                         edgecolor='black', linewidth=0.4, hatch=tier_hatches[tier])
        axis.set_title(f'{task} – {cluster}')
        axis.set_xticks(x, round_targets)
        axis.set_xlabel('Round')
        axis.grid(axis='y', alpha=0.25)
    for axis in axes[:, 0]:
        axis.set_ylabel('Accuracy (%)')
    method_handles = [plt.Rectangle((0, 0), 1, 1, color=METHOD_COLORS[m]) for m in methods]
    tier_handles = [plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor='black',
                                  hatch=tier_hatches[t]) for t in tiers]
    fig.legend(method_handles + tier_handles, methods + tiers, loc='upper center',
               ncol=6, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.91))
    save_figure(fig, 'Figure16', 'figure16_heterogeneous_devices')
    display_figure(fig)
else:
    print('Figure 16: disabled')

## Execution summary

In [ ]:
flags = {
    'Figure10': RUN_FIGURE_10,
    'Figure12': RUN_FIGURE_12,
    'Table3': RUN_TABLE_3,
    'Figure14': RUN_FIGURE_14,
    'Figure16': RUN_FIGURE_16,
    'Figure19': RUN_FIGURE_19,
    'Figure20': RUN_FIGURE_20,
    'Figure21': RUN_FIGURE_21,
    'Figure22': RUN_FIGURE_22,
}
summary = pd.DataFrame([
    {
        'claim': claim,
        'enabled': enabled,
        'raw_directory_exists': (RESULTS_ROOT / claim / 'raw').is_dir(),
        'output_directory_exists': (RESULTS_ROOT / claim / 'output').is_dir(),
    }
    for claim, enabled in flags.items()
])
display(summary)